# Validation Set — Batch Inference
Runs all (or a random subset of) validation samples through the full pipeline
(backbone → transformer → coarse matching → morph decoder → Sinkhorn → LGR).

Ground truth is available for every sample: `gt_z`, `transform`.

**Metrics computed per sample:**
- `RRE` — relative rotation error (degrees)
- `RTE` — relative translation error (metres)
- `RMSE` — point cloud RMSE after predicted alignment
- `RR` — registration recall (1 if RMSE < threshold)
- `PIR` — patch inlier ratio
- `IR` — inlier ratio
- `morph_loss` — morphing loss (pred_z vs gt_z)
- `CD` — one-sided Chamfer: aligned src → morphed ref
- `CD_mean` — one-sided Chamfer: aligned src → mean ref

**Workflow:**
1. Load model from checkpoint
2. Run all val samples → store in `results`
3. Summary table + distribution plots
4. Pick a sample by index → qualitative visualisation

In [ ]:
# ── USER CONFIG ───────────────────────────────────────────────────────────────
SNAPSHOT       = 'epoch-35.pth.tar'  # checkpoint filename inside output/.../snapshots/
DEVICE         = 'cuda'

# Set to None to run all val samples, or an integer to randomly sample that many.
MAX_SAMPLES    = None

# Val-loader augmentation settings (scale=1.0, subsample=1.0 → no aug, deterministic)
VAL_AUG_SCALE     = 1.0
VAL_AUG_SUBSAMPLE = 1.0
# ─────────────────────────────────────────────────────────────────────────────

In [ ]:
import os, sys, random
from functools import partial

EXP_DIR  = os.path.dirname(os.path.abspath('__file__'))
ROOT_DIR = os.path.dirname(os.path.dirname(EXP_DIR))
sys.path.insert(0, EXP_DIR)
sys.path.insert(0, ROOT_DIR)
os.chdir(EXP_DIR)

import torch
import numpy as np
import plotly.graph_objects as go
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from io import BytesIO
from IPython.display import Image, display

from geotransformer.utils.data import registration_collate_fn_stack_mode
from geotransformer.modules.ops.transformation import apply_transform

from config_dowsampled import make_cfg
from dataset import train_valid_data_loader
from model import create_model
from loss import OverallLoss, Evaluator

print('EXP_DIR :', EXP_DIR)
print('ROOT_DIR:', ROOT_DIR)

In [ ]:
cfg = make_cfg()

_, val_loader, neighbor_limits = train_valid_data_loader(
    cfg, distributed=False,
    val_aug_scale=VAL_AUG_SCALE,
    val_aug_subsample=VAL_AUG_SUBSAMPLE,
)
print('neighbor_limits:', neighbor_limits)
print('val samples    :', len(val_loader.dataset))

collate_fn = partial(
    registration_collate_fn_stack_mode,
    num_stages=cfg.backbone.num_stages,
    voxel_size=cfg.backbone.init_voxel_size,
    search_radius=cfg.backbone.init_radius,
    neighbor_limits=neighbor_limits,
    precompute_data=False,
)

SNAP_DIR = os.path.join(
    ROOT_DIR, 'output',
    'geotransformer.facesdownsampledfixed.stage4.gse.k3.max.oacl.stage2.sinkhorn.vl',
    'snapshots',
)
ckpt_path = os.path.join(SNAP_DIR, SNAPSHOT)

model = create_model(cfg).to(DEVICE)
model.neighbor_limits = neighbor_limits

ckpt = torch.load(ckpt_path, map_location=DEVICE)
sd   = ckpt.get('model', ckpt)
sd   = {k.replace('module.', ''): v for k, v in sd.items()}

missing = set(model.state_dict()) - set(sd)
extra   = set(sd) - set(model.state_dict())
if missing: print(f'[warn] missing keys (random init): {len(missing)}')
if extra:   print(f'[info] extra keys ignored         : {len(extra)}')

model.load_state_dict(sd, strict=False)
model.eval()

loss_fn   = OverallLoss(cfg).to(DEVICE)
evaluator = Evaluator(cfg).to(DEVICE)

print(f'Loaded: {SNAPSHOT}  (epoch {ckpt.get("epoch", "?")})')
print(f'Keys matched: {len(set(sd) & set(model.state_dict()))} / {len(model.state_dict())}')

with torch.no_grad():
    mean_ref = model.generate_reference_geometry(torch.zeros(32, cfg.model.num_pca_components, device=DEVICE))
print(f'Mean ref: {mean_ref.shape}')

In [ ]:
def chamfer_distance(A, B, device=DEVICE, single_sided=False):
    """Chamfer distance (mean of NN distances, metres)."""
    if isinstance(A, np.ndarray):
        A = torch.from_numpy(A).float().to(device)
    if isinstance(B, np.ndarray):
        B = torch.from_numpy(B).float().to(device)
    chunk = 2048
    dists_AB = [torch.cdist(A[i:i+chunk], B).min(dim=1).values for i in range(0, A.shape[0], chunk)]
    dists_BA = [torch.cdist(B[i:i+chunk], A).min(dim=1).values for i in range(0, B.shape[0], chunk)]
    if single_sided:
        chamfer = torch.cat(dists_AB).mean().item()
        std     = torch.cat(dists_AB).std().item()
    else:
        chamfer = 0.5 * (torch.cat(dists_AB).mean().item() + torch.cat(dists_BA).mean().item())
        std     = 0.5 * (torch.cat(dists_AB).std().item()  + torch.cat(dists_BA).std().item())
    return chamfer, std


def pcd_trace(pts, color, name, size=2, opacity=0.7):
    if isinstance(pts, torch.Tensor):
        pts = pts.detach().cpu().numpy()
    return go.Scatter3d(
        x=pts[:, 0], y=pts[:, 1], z=pts[:, 2],
        mode='markers',
        marker=dict(size=size, color=color, opacity=opacity),
        name=name,
    )


def show_pcd(traces, title='', height=600):
    fig = go.Figure(data=traces)
    fig.update_layout(
        title=title, height=height,
        scene=dict(aspectmode='data'),
        legend=dict(itemsizing='constant'),
        margin=dict(l=0, r=0, b=0, t=40),
    )
    fig.show()


def plot_chamfer_heatmap(src_pts, ref_pts, title='One-sided Chamfer heatmap',
                         colorscale='Turbo', point_size=2, dist_scale=10):
    if isinstance(src_pts, np.ndarray):
        src = torch.from_numpy(src_pts).float()
    else:
        src = src_pts.detach().float().cpu()
    if isinstance(ref_pts, np.ndarray):
        ref = torch.from_numpy(ref_pts).float()
    else:
        ref = ref_pts.detach().float().cpu()

    chunk = 2048
    nn_dists = [torch.cdist(src[i:i+chunk], ref).min(dim=1).values for i in range(0, src.shape[0], chunk)]
    nn_dist  = torch.cat(nn_dists, dim=0) * dist_scale
    cd_one   = nn_dist.mean().item()

    fig = go.Figure(data=[go.Scatter3d(
        x=src[:, 0].numpy(), y=src[:, 1].numpy(), z=src[:, 2].numpy(),
        mode='markers',
        marker=dict(size=point_size, color=nn_dist.numpy(),
                    colorscale=colorscale, showscale=True,
                    colorbar=dict(title='NN dist (cm)'), opacity=0.9),
        name=f'src (CD={cd_one:.4f} cm)',
    )])
    full_title = (f"{title}<br>"
                  f"<span style='font-size:12px;color:gray;'>"
                  f"Chamfer (one-sided) = {cd_one:.4f} cm</span>")
    fig.update_layout(title=full_title, scene=dict(aspectmode='data'),
                      margin=dict(l=0, r=0, b=0, t=40))
    fig.show()

---
## Run all val samples (inference + metrics)

In [ ]:
dataset   = val_loader.dataset
n_total   = len(dataset)

if MAX_SAMPLES is None:
    indices = list(range(n_total))
else:
    indices = sorted(random.sample(range(n_total), min(MAX_SAMPLES, n_total)))

print(f'Running {len(indices)} / {n_total} val samples')

results = {}  # idx -> result dict

for idx in indices:
    try:
        raw       = dataset[idx]
        data_dict = collate_fn([raw])
        for k, v in data_dict.items():
            if isinstance(v, torch.Tensor):
                data_dict[k] = v.to(DEVICE)
            elif isinstance(v, list) and v and isinstance(v[0], torch.Tensor):
                data_dict[k] = [t.to(DEVICE) for t in v]

        with torch.no_grad():
            out = model(data_dict)

        loss_dict = loss_fn(out, data_dict, epoch=999, mode='val')
        eval_dict = evaluator(out, data_dict)

        morphed_ref = out['morphed_full']
        est_tf      = out['estimated_transform']
        gt_tf       = data_dict['transform']
        src_model   = out['src_points']
        src_aligned = apply_transform(src_model, est_tf)
        src_gt_aln  = apply_transform(src_model, gt_tf)

        cd_pred, cd_pred_std   = chamfer_distance(src_aligned, morphed_ref, single_sided=True)
        cd_mean, cd_mean_std   = chamfer_distance(src_aligned, mean_ref,    single_sided=True)
        cd_gt,   cd_gt_std     = chamfer_distance(src_gt_aln,  morphed_ref, single_sided=True)

        # ── Morphing metrics ───────────────────────────────────────────────────
        z_pred       = out['z_coefficients'].detach().cpu().float()   # [32, n_comp]
        recon_gt_pts = out['recon_gt_points'].cpu().float()           # [N_verts, 3]
        morph_mse    = float(((morphed_ref.cpu().float() - recon_gt_pts) ** 2).mean())

        results[idx] = {
            'idx'          : idx,
            'scene_name'   : raw.get('scene_name', ''),
            'n_src'        : src_model.shape[0],
            # registration metrics
            'RRE'          : eval_dict['RRE'].item(),
            'RTE'          : eval_dict['RTE'].item(),
            'RMSE'         : eval_dict['RMSE'].item(),
            'RR'           : eval_dict['RR'].item(),
            'PIR'          : eval_dict['PIR'].item(),
            'IR'           : eval_dict['IR'].item(),
            # losses
            'morph_loss'   : loss_dict['m_loss'].item(),
            'coarse_loss'  : loss_dict['c_loss'].item(),
            'fine_loss'    : loss_dict['f_loss'].item(),
            # chamfer
            'chamfer'      : cd_pred,
            'chamfer_std'  : cd_pred_std,
            'chamfer_mean' : cd_mean,
            'chamfer_mean_std': cd_mean_std,
            'chamfer_gt'   : cd_gt,
            'chamfer_gt_std': cd_gt_std,
            # morphing
            'z_pred'       : z_pred.numpy(),        # [32, n_comp]
            'morph_mse'    : morph_mse,             # scalar (m²)
            # point clouds (numpy, cpu)
            'src_raw'      : raw['src_points'],
            'src_aligned'  : src_aligned.cpu().numpy(),
            'src_gt_aln'   : src_gt_aln.cpu().numpy(),
            'morphed_ref'  : morphed_ref.cpu().numpy(),
            'ref_raw'      : raw['ref_points'],
            'mean_ref'     : mean_ref.cpu().numpy(),
            'error'        : None,
        }
        r = results[idx]
        print(f'[{idx:4d}] {r["scene_name"]:20s}  '
              f'n={r["n_src"]:5d}  '
              f'RRE={r["RRE"]:6.2f}°  RTE={r["RTE"]:.4f}m  RMSE={r["RMSE"]:.4f}m  '
              f'RR={r["RR"]:3.0f}  '
              f'CD={r["chamfer"]:.4f}  morph_mse={r["morph_mse"]:.6f}')

    except Exception as e:
        import traceback
        print(f'[{idx:4d}]  ERROR: {e}')
        results[idx] = {
            'idx': idx, 'error': str(e),
            'RRE': float('nan'), 'RTE': float('nan'), 'RMSE': float('nan'),
            'RR': float('nan'), 'PIR': float('nan'), 'IR': float('nan'),
            'morph_loss': float('nan'), 'chamfer': float('nan'), 'chamfer_mean': float('nan'),
            'morph_mse': float('nan'),
        }

ok = [r for r in results.values() if r['error'] is None]
print(f'\nDone — {len(ok)}/{len(results)} succeeded')

---
## Summary statistics

In [ ]:
import pandas as pd

rows = []
for idx, r in results.items():
    rows.append({
        'idx'         : r.get('idx', idx),
        'scene_name'  : r.get('scene_name', ''),
        'n_src'       : r.get('n_src', np.nan),
        'RRE'         : r.get('RRE', np.nan),
        'RTE'         : r.get('RTE', np.nan),
        'RMSE'        : r.get('RMSE', np.nan),
        'RR'          : r.get('RR', np.nan),
        'PIR'         : r.get('PIR', np.nan),
        'IR'          : r.get('IR', np.nan),
        'morph_loss'  : r.get('morph_loss', np.nan),
        'chamfer'     : r.get('chamfer', np.nan),
        'chamfer_mean': r.get('chamfer_mean', np.nan),
        'error'       : r.get('error', None),
    })

df = pd.DataFrame(rows).set_index('idx').sort_index()
df

In [ ]:
uhm_to_mm = 100  # UHM model ~2.5m scale; multiply by 100 → cm-scale

metric_keys = ['RRE', 'RTE', 'RMSE', 'RR', 'PIR', 'IR', 'morph_loss', 'chamfer', 'chamfer_mean']
arrays = {k: np.array([r[k] for r in ok]) for k in metric_keys}

print('=' * 80)
print(f'  Val batch — {len(ok)} samples  (checkpoint: {SNAPSHOT})')
print('=' * 80)
print(f'  {"Metric":<18}  {"Mean":>10}  {"Median":>10}  {"Std":>10}  {"Min":>10}  {"Max":>10}')
print('-' * 80)
for k, arr in arrays.items():
    print(f'  {k:<18}  {arr.mean():>10.4f}  {np.median(arr):>10.4f}  '
          f'{arr.std():>10.4f}  {arr.min():>10.4f}  {arr.max():>10.4f}')
print('=' * 80)
print(f'  Registration Recall (RR): {arrays["RR"].mean()*100:.1f}%')
print(f'  Chamfer (mm)            : mean={arrays["chamfer"].mean()*uhm_to_mm:.4f}  '
      f'median={np.median(arrays["chamfer"])*uhm_to_mm:.4f}')
print(f'  Chamfer mean-ref (mm)   : mean={arrays["chamfer_mean"].mean()*uhm_to_mm:.4f}  '
      f'median={np.median(arrays["chamfer_mean"])*uhm_to_mm:.4f}')

In [ ]:
# Distribution histograms for registration metrics
fig = go.Figure()
for key, color in [('RRE', 'steelblue'), ('RTE', 'tomato'), ('RMSE', 'seagreen')]:
    fig.add_trace(go.Histogram(
        x=arrays[key], name=key, opacity=0.6,
        marker_color=color, nbinsx=40,
    ))
fig.update_layout(
    barmode='overlay',
    title='Distribution of registration errors (RRE °, RTE m, RMSE m)',
    xaxis_title='Error', yaxis_title='Frequency',
)
fig.show()

In [ ]:
# Distribution histograms for Chamfer distances
fig = go.Figure()
fig.add_trace(go.Histogram(
    x=arrays['chamfer'], name='CD (aligned ↔ morphed_ref)',
    opacity=0.6, marker_color='steelblue', nbinsx=40,
))
fig.add_trace(go.Histogram(
    x=arrays['chamfer_mean'], name='CD (aligned ↔ mean_ref)',
    opacity=0.6, marker_color='tomato', nbinsx=40,
))
fig.update_layout(
    barmode='overlay',
    title='Distribution of Chamfer distances',
    xaxis_title='Chamfer distance (m)', yaxis_title='Frequency',
)
fig.show()

In [ ]:
# Scatter: RRE vs Chamfer — useful to see if bad registrations drive high CD
fig = go.Figure(data=go.Scatter(
    x=arrays['RRE'], y=arrays['chamfer'],
    mode='markers',
    marker=dict(size=5, color=arrays['RR'], colorscale='RdYlGn',
                showscale=True, colorbar=dict(title='RR')),
    text=[str(r['idx']) for r in ok],
))
fig.update_layout(
    title='RRE vs Chamfer distance (colour = RR)',
    xaxis_title='RRE (°)', yaxis_title='Chamfer (m)',
)
fig.show()

In [ ]:
# Bar chart: Chamfer per sample (sorted by Chamfer ascending)
ok_sorted = sorted(ok, key=lambda r: r['chamfer'])
names = [str(r['idx']) for r in ok_sorted]
vals  = [r['chamfer'] * uhm_to_mm for r in ok_sorted]

fig, ax = plt.subplots(figsize=(max(6, len(ok_sorted) * 0.15), 4))
ax.bar(range(len(names)), vals, color='steelblue', edgecolor='none', linewidth=0)
ax.axhline(np.mean(vals),   color='tomato',   linestyle='--', linewidth=1.2,
           label=f'mean={np.mean(vals):.4f} mm')
ax.axhline(np.median(vals), color='seagreen', linestyle=':',  linewidth=1.2,
           label=f'median={np.median(vals):.4f} mm')
ax.set_xlabel('Sample (sorted by CD)')
ax.set_ylabel('Chamfer distance (mm)')
ax.set_title('Chamfer distance per val sample (src_aligned ↔ morphed_ref)')
ax.set_xticks([])
ax.legend(fontsize=8)
plt.tight_layout()

buf = BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

In [ ]:
# Bar chart: RRE per sample (sorted by RRE ascending)
ok_sorted_rre = sorted(ok, key=lambda r: r['RRE'])
rre_vals = [r['RRE'] for r in ok_sorted_rre]

fig, ax = plt.subplots(figsize=(max(6, len(ok_sorted_rre) * 0.15), 4))
ax.bar(range(len(rre_vals)), rre_vals, color='steelblue', edgecolor='none')
ax.axhline(np.mean(rre_vals),   color='tomato',   linestyle='--', linewidth=1.2,
           label=f'mean={np.mean(rre_vals):.3f}°')
ax.axhline(np.median(rre_vals), color='seagreen', linestyle=':',  linewidth=1.2,
           label=f'median={np.median(rre_vals):.3f}°')
ax.set_xlabel('Sample (sorted by RRE)')
ax.set_ylabel('RRE (°)')
ax.set_title('RRE per val sample')
ax.set_xticks([])
ax.legend(fontsize=8)
plt.tight_layout()

buf = BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

---
## Qualitative visualisation — pick a sample
Set `VIZ_IDX` to any index that appears in `results`, then run the cells below.

In [ ]:
VIZ_IDX = 0   # ← change to any index from the results dict

print('Available indices (first 20):')
for idx, r in list(results.items())[:20]:
    tag    = (f"RRE={r['RRE']:.2f}° RTE={r['RTE']:.4f}m CD={r['chamfer']:.4f}"
              if r['error'] is None else f"ERROR: {r['error']}")
    marker = '  ◀' if idx == VIZ_IDX else ''
    print(f'  [{idx:4d}]  {tag}{marker}')

In [ ]:
r = results.get(VIZ_IDX)
assert r is not None,    f'Index {VIZ_IDX} not found in results'
assert r['error'] is None, f'Sample {VIZ_IDX} failed: {r["error"]}'

print(f'Visualising idx={VIZ_IDX}  scene={r["scene_name"]}  n_src={r["n_src"]}')
print(f'  RRE={r["RRE"]:.3f}°  RTE={r["RTE"]:.4f}m  RMSE={r["RMSE"]:.4f}m  '
      f'RR={r["RR"]:.0f}  morph_loss={r["morph_loss"]:.6f}')
print(f'  CD={r["chamfer"]:.4f}m  CD_mean={r["chamfer_mean"]:.4f}m')

# Before alignment
show_pcd(
    [pcd_trace(r['ref_raw'],  'steelblue', 'GT ref (morphed)'),
     pcd_trace(r['src_raw'],  'tomato',    f'src raw ({r["n_src"]} pts)')],
    title=f'[{VIZ_IDX}] Before alignment',
)

# Predicted alignment vs mean ref
show_pcd(
    [pcd_trace(r['mean_ref'],    'steelblue', 'mean ref'),
     pcd_trace(r['src_aligned'], 'orange',    f'aligned (pred)  CD_mean={r["chamfer_mean"]:.4f}m')],
    title=f'[{VIZ_IDX}] Predicted alignment — vs mean ref',
)

# Predicted alignment vs morphed ref
show_pcd(
    [pcd_trace(r['morphed_ref'], 'steelblue', 'morphed ref (pred z)'),
     pcd_trace(r['src_aligned'], 'orange',    f'aligned (pred)  CD={r["chamfer"]:.4f}m')],
    title=f'[{VIZ_IDX}] Predicted alignment — vs morphed ref',
)

# GT alignment vs morphed ref
show_pcd(
    [pcd_trace(r['morphed_ref'], 'steelblue', 'morphed ref (pred z)'),
     pcd_trace(r['src_gt_aln'],  'seagreen',  f'aligned (GT tf)  CD_gt={r["chamfer_gt"]:.4f}m')],
    title=f'[{VIZ_IDX}] GT alignment — vs morphed ref',
)

In [ ]:
# Morphing quality: mean ref vs morphed ref
show_pcd(
    [pcd_trace(r['mean_ref'],    'steelblue', 'mean ref'),
     pcd_trace(r['morphed_ref'], 'orange',    'morphed ref (pred z)')],
    title=f'[{VIZ_IDX}] Morph quality',
)

In [ ]:
# Chamfer heatmap: aligned src coloured by distance to morphed ref
plot_chamfer_heatmap(
    r['src_aligned'], r['morphed_ref'],
    title=f'[{VIZ_IDX}] Aligned src — distance to morphed ref',
)

# Chamfer heatmap: morphed ref coloured by distance to mean ref
plot_chamfer_heatmap(
    r['morphed_ref'], r['mean_ref'],
    title=f'[{VIZ_IDX}] Morphed ref — distance to mean ref',
)

In [ ]:
# Re-run inference for the selected sample to get per-patch z coefficients
raw_viz   = dataset[VIZ_IDX]
dd_viz    = collate_fn([raw_viz])
for k, v in dd_viz.items():
    if isinstance(v, torch.Tensor):
        dd_viz[k] = v.to(DEVICE)
    elif isinstance(v, list) and v and isinstance(v[0], torch.Tensor):
        dd_viz[k] = [t.to(DEVICE) for t in v]

with torch.no_grad():
    out_viz = model(dd_viz)

z = out_viz['z_coefficients'].detach().cpu().float()
if z.ndim == 3 and z.shape[0] == 1:
    z = z[0]
if z.ndim != 2:
    raise ValueError(f'Unexpected z_coefficients shape: {tuple(z.shape)}')
if z.shape[0] != 32 and z.shape[1] == 32:
    z = z.T

n_patches, n_coeffs = z.shape
print(f'z_coefficients shape: {n_patches} patches × {n_coeffs} coefficients')

fig, axes = plt.subplots(8, 4, figsize=(20, 24), sharex=True, sharey=True)
axes = axes.ravel()
x = np.arange(n_coeffs)

for i in range(n_patches):
    ax = axes[i]
    ax.bar(x, z[i].numpy(), color='steelblue', width=0.8)
    ax.set_title(f'Patch {i+1}', fontsize=9)
    ax.axhline(0.0, color='black', linewidth=0.6)
    ax.tick_params(axis='both', labelsize=7)
    ax.set_ylim(-1.0, 1.0)

for ax in axes[-4:]:
    ax.set_xlabel('Coeff idx', fontsize=8)
for k in range(0, n_patches, 4):
    axes[k].set_ylabel('Value', fontsize=8)

fig.suptitle(f'Z coefficients per patch — val idx {VIZ_IDX}', fontsize=14, y=0.995)
plt.tight_layout(rect=[0, 0, 1, 0.985])

buf = BytesIO()
fig.savefig(buf, format='png', dpi=140, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

---
## Error analysis: shape deviation, height, width

For each succeeded val sample we collect:
- **z_rms_k** — RMS of `gt_z[:, k]` across patches for component k=1..5  
  (zero-mean PCA → RMS measures how far sample k deviates from the mean face)
- **face_height** — bounding-box Y-range of the reference point cloud (metres)
- **face_width**  — bounding-box X-range of the reference point cloud (metres)

In [ ]:
# ── Enrich results with gt_z shape descriptors + face geometry ────────────────
N_COMP_PLOT = 5  # analyse first N PCA components

for idx, r in results.items():
    if r['error'] is not None:
        continue

    raw   = dataset[idx]
    gt_z  = raw['gt_z']
    if isinstance(gt_z, torch.Tensor):
        gt_z = gt_z.float()
    else:
        gt_z = torch.tensor(gt_z, dtype=torch.float32)

    # gt_z: (n_patches, n_components)  — squeeze any batch dim
    if gt_z.ndim == 3 and gt_z.shape[0] == 1:
        gt_z = gt_z[0]
    if gt_z.ndim == 1:
        gt_z = gt_z.unsqueeze(0)    # (1, n_comp)

    n_comp = gt_z.shape[1]

    # RMS across patches per component  (shape deviation from mean face)
    r['z_rms'] = [
        float(gt_z[:, k].pow(2).mean().sqrt())
        for k in range(min(N_COMP_PLOT, n_comp))
    ]

    # Per-patch × per-component squared error between pred and gt z
    z_pred = torch.tensor(r['z_pred'], dtype=torch.float32)   # [32, n_comp_pred]
    n_comp_common = min(z_pred.shape[1], n_comp)
    z_err_sq = (z_pred[:, :n_comp_common] - gt_z[:, :n_comp_common]) ** 2  # [32, n_comp_common]
    r['z_err_sq']    = z_err_sq.numpy()           # per-patch per-component squared error
    r['z_mae_comp']  = z_err_sq.sqrt().mean(dim=0).numpy()  # mean abs err per component [n_comp]
    r['z_mae_patch'] = z_err_sq.sqrt().mean(dim=1).numpy()  # mean abs err per patch    [32]

    # Face geometry from the reference (morphed) point cloud
    pts = r['ref_raw']                          # numpy (N, 3)
    r['face_height'] = float(pts[:, 1].max() - pts[:, 1].min())
    r['face_width']  = float(pts[:, 0].max() - pts[:, 0].min())

ok_enriched = [r for r in results.values() if r['error'] is None and 'z_rms' in r]

# Shared metric arrays used by all analysis cells below
rre_arr = np.array([r['RRE'] for r in ok_enriched])
rte_arr = np.array([r['RTE'] for r in ok_enriched])
rr_arr  = np.array([r['RR']  for r in ok_enriched])

print(f'Enriched {len(ok_enriched)} samples')
print(f'  z_rms[0] range : {min(r["z_rms"][0] for r in ok_enriched):.4f} – '
      f'{max(r["z_rms"][0] for r in ok_enriched):.4f}')
print(f'  face_height (m): {min(r["face_height"] for r in ok_enriched):.4f} – '
      f'{max(r["face_height"] for r in ok_enriched):.4f}')
print(f'  face_width  (m): {min(r["face_width"]  for r in ok_enriched):.4f} – '
      f'{max(r["face_width"]  for r in ok_enriched):.4f}')
print(f'  morph_mse  range: {min(r["morph_mse"] for r in ok_enriched):.6f} – '
      f'{max(r["morph_mse"] for r in ok_enriched):.6f}')

In [ ]:
# ── Heatmap: mean z MAE per (patch, component) across all val samples ─────────
# Rows = 32 patches, Columns = all components
# Shows where the model struggles: which patches / which shape modes are hardest.

z_mae_grid = np.stack([r['z_mae_comp'] for r in ok_enriched], axis=0)   # [N_samples, n_comp]
z_err_grid = np.stack([r['z_err_sq']   for r in ok_enriched], axis=0)   # [N_samples, 32, n_comp]

n_patches = z_err_grid.shape[1]
n_comp    = z_err_grid.shape[2]

mean_mae_patch_comp = np.sqrt(z_err_grid.mean(axis=0))  # [32, n_comp] — RMSE across samples

fig, axes = plt.subplots(1, 2, figsize=(18, 7))

# ── Left: full heatmap [patch × component] ──
ax = axes[0]
im = ax.imshow(mean_mae_patch_comp, aspect='auto', cmap='viridis',
               origin='upper', interpolation='nearest')
ax.set_xlabel('PCA component index', fontsize=10)
ax.set_ylabel('Patch index', fontsize=10)
ax.set_title('Mean z MAE per patch × component\n(averaged over val set)', fontsize=11)
plt.colorbar(im, ax=ax, label='MAE (z units)')

# ── Right: marginal bar charts ──
ax2 = axes[1]
# Mean MAE per component (averaged across patches)
mean_mae_per_comp  = mean_mae_patch_comp.mean(axis=0)   # [n_comp]
mean_mae_per_patch = mean_mae_patch_comp.mean(axis=1)   # [32]

ax2.bar(np.arange(n_comp),  mean_mae_per_comp,  color='steelblue', alpha=0.8, label='per component')
ax2.set_xlabel('Component index', fontsize=10)
ax2.set_ylabel('Mean MAE across patches', fontsize=10)
ax2.set_title('z MAE per component\n(mean over patches & val set)', fontsize=11)

# inset: per-patch marginal
ax3 = ax2.inset_axes([0.55, 0.45, 0.42, 0.50])
ax3.barh(np.arange(n_patches), mean_mae_per_patch[::-1],
         color='tomato', alpha=0.8)
ax3.set_yticks([])
ax3.set_xlabel('MAE', fontsize=8)
ax3.set_title('Per patch', fontsize=8)
ax3.invert_yaxis()

plt.tight_layout()
buf = BytesIO()
fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

print(f'\nComponent with highest MAE : {mean_mae_per_comp.argmax()} '
      f'({mean_mae_per_comp.max():.5f})')
print(f'Component with lowest  MAE : {mean_mae_per_comp.argmin()} '
      f'({mean_mae_per_comp.min():.5f})')
print(f'Patch with highest MAE     : {mean_mae_per_patch.argmax()} '
      f'({mean_mae_per_patch.max():.5f})')
print(f'Patch with lowest  MAE     : {mean_mae_per_patch.argmin()} '
      f'({mean_mae_per_patch.min():.5f})')

In [ ]:
# ── Overall morph MSE: summary stats + distribution + scatter vs reg errors ───
morph_mse_arr = np.array([r['morph_mse'] for r in ok_enriched])
morph_mse_mm2 = morph_mse_arr * (100 ** 2)   # convert m² → cm² for readability

print('=' * 60)
print(f'  Morph MSE (predicted vs GT morph) — {len(ok_enriched)} samples')
print('=' * 60)
print(f'  mean   : {morph_mse_arr.mean():.6f} m²  ({morph_mse_mm2.mean():.4f} cm²)')
print(f'  median : {np.median(morph_mse_arr):.6f} m²  ({np.median(morph_mse_mm2):.4f} cm²)')
print(f'  std    : {morph_mse_arr.std():.6f} m²')
print(f'  min    : {morph_mse_arr.min():.6f} m²')
print(f'  max    : {morph_mse_arr.max():.6f} m²')
print(f'  RMSE   : {np.sqrt(morph_mse_arr).mean()*100:.4f} cm')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# ── Distribution ──
ax = axes[0]
ax.hist(morph_mse_mm2, bins=40, color='mediumpurple', edgecolor='none', alpha=0.8)
ax.axvline(morph_mse_mm2.mean(),   color='tomato',   linestyle='--', linewidth=1.3,
           label=f'mean={morph_mse_mm2.mean():.3f}')
ax.axvline(np.median(morph_mse_mm2), color='seagreen', linestyle=':',  linewidth=1.3,
           label=f'median={np.median(morph_mse_mm2):.3f}')
ax.set_xlabel('Morph MSE (cm²)', fontsize=10)
ax.set_ylabel('Frequency', fontsize=10)
ax.set_title('Morph MSE distribution', fontsize=11)
ax.legend(fontsize=8)

# ── Morph MSE vs RRE ──
ax = axes[1]
sc = ax.scatter(rre_arr, morph_mse_mm2, c=rr_arr, cmap='RdYlGn',
                vmin=0, vmax=1, s=20, alpha=0.75, edgecolors='none')
m, b = np.polyfit(rre_arr, morph_mse_mm2, 1)
xs = np.linspace(rre_arr.min(), rre_arr.max(), 100)
ax.plot(xs, m * xs + b, color='royalblue', linewidth=1.3,
        label=f'r={np.corrcoef(rre_arr, morph_mse_mm2)[0,1]:.3f}')
ax.set_xlabel('RRE (°)', fontsize=10)
ax.set_ylabel('Morph MSE (cm²)', fontsize=10)
ax.set_title('Morph MSE vs RRE', fontsize=11)
ax.legend(fontsize=8)
plt.colorbar(sc, ax=ax, label='RR')

# ── Morph MSE vs RTE ──
ax = axes[2]
sc = ax.scatter(rte_arr, morph_mse_mm2, c=rr_arr, cmap='RdYlGn',
                vmin=0, vmax=1, s=20, alpha=0.75, edgecolors='none')
m2, b2 = np.polyfit(rte_arr, morph_mse_mm2, 1)
xs2 = np.linspace(rte_arr.min(), rte_arr.max(), 100)
ax.plot(xs2, m2 * xs2 + b2, color='tomato', linewidth=1.3,
        label=f'r={np.corrcoef(rte_arr, morph_mse_mm2)[0,1]:.3f}')
ax.set_xlabel('RTE (m)', fontsize=10)
ax.set_ylabel('Morph MSE (cm²)', fontsize=10)
ax.set_title('Morph MSE vs RTE', fontsize=11)
ax.legend(fontsize=8)
plt.colorbar(sc, ax=ax, label='RR')

plt.suptitle('Overall morph MSE: predicted morphed geometry vs GT morphed geometry\n'
             '(colour = registration recall)', fontsize=12)
plt.tight_layout()

buf = BytesIO()
fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

In [ ]:
# ── Error vs deviation from mean (first 5 PCA components) ────────────────────
fig, axes = plt.subplots(2, N_COMP_PLOT, figsize=(4 * N_COMP_PLOT, 8))

for k in range(N_COMP_PLOT):
    z_k = np.array([r['z_rms'][k] for r in ok_enriched])

    # ── RRE row ──
    ax = axes[0, k]
    sc = ax.scatter(z_k, rre_arr, c=rr_arr, cmap='RdYlGn',
                    vmin=0, vmax=1, s=18, alpha=0.75, edgecolors='none')
    m, b = np.polyfit(z_k, rre_arr, 1)
    xs = np.linspace(z_k.min(), z_k.max(), 100)
    ax.plot(xs, m * xs + b, color='royalblue', linewidth=1.2,
            label=f'slope={m:.2f}')
    corr = np.corrcoef(z_k, rre_arr)[0, 1]
    ax.set_title(f'Component {k+1}\nr={corr:.3f}', fontsize=10)
    ax.set_xlabel(f'z_rms[{k+1}]', fontsize=9)
    if k == 0:
        ax.set_ylabel('RRE (°)', fontsize=9)
    ax.legend(fontsize=7)

    # ── RTE row ──
    ax = axes[1, k]
    sc = ax.scatter(z_k, rte_arr, c=rr_arr, cmap='RdYlGn',
                    vmin=0, vmax=1, s=18, alpha=0.75, edgecolors='none')
    m2, b2 = np.polyfit(z_k, rte_arr, 1)
    ax.plot(xs, m2 * xs + b2, color='tomato', linewidth=1.2,
            label=f'slope={m2:.4f}')
    corr2 = np.corrcoef(z_k, rte_arr)[0, 1]
    ax.set_title(f'Component {k+1}\nr={corr2:.3f}', fontsize=10)
    ax.set_xlabel(f'z_rms[{k+1}]', fontsize=9)
    if k == 0:
        ax.set_ylabel('RTE (m)', fontsize=9)
    ax.legend(fontsize=7)

cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(sc, cax=cbar_ax, label='RR')

fig.suptitle('RRE / RTE vs PCA shape deviation (z_rms per component)\n'
             'colour = registration recall', fontsize=13)
plt.tight_layout(rect=[0, 0, 0.91, 0.96])

buf = BytesIO()
fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))

In [ ]:
# ── Error vs face height and width ───────────────────────────────────────────
height_arr = np.array([r['face_height'] for r in ok_enriched])
width_arr  = np.array([r['face_width']  for r in ok_enriched])

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for col, (x_arr, xlabel) in enumerate([
        (height_arr, 'Face height (m)'),
        (width_arr,  'Face width (m)'),
]):
    for row, (y_arr, ylabel, color) in enumerate([
            (rre_arr, 'RRE (°)',  'royalblue'),
            (rte_arr, 'RTE (m)',  'tomato'),
    ]):
        ax = axes[row, col]
        sc = ax.scatter(x_arr, y_arr, c=rr_arr, cmap='RdYlGn',
                        vmin=0, vmax=1, s=20, alpha=0.75, edgecolors='none')
        m, b = np.polyfit(x_arr, y_arr, 1)
        xs = np.linspace(x_arr.min(), x_arr.max(), 100)
        ax.plot(xs, m * xs + b, color=color, linewidth=1.4,
                label=f'slope={m:.3f}')
        corr = np.corrcoef(x_arr, y_arr)[0, 1]
        ax.set_title(f'{ylabel} vs {xlabel.split()[1]}\nr={corr:.3f}',
                     fontsize=10)
        ax.set_xlabel(xlabel, fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.legend(fontsize=8)

# shared colorbar
cbar_ax = fig.add_axes([0.92, 0.15, 0.015, 0.7])
fig.colorbar(sc, cax=cbar_ax, label='RR')

fig.suptitle('RRE / RTE vs face geometry (bounding-box height & width)\n'
             'colour = registration recall', fontsize=13)
plt.tight_layout(rect=[0, 0, 0.91, 0.96])

buf = BytesIO()
fig.savefig(buf, format='png', dpi=130, bbox_inches='tight')
plt.close(fig)
buf.seek(0)
display(Image(buf.read()))